# LLM Excercise

This notebook collects the first-topic BPE exercises from the assignment into one place.

Each exercise section uses the exact assignment name, and includes:
- a short description of the task
- an explanation of what the exercise is testing
- pseudocode or a structured attack plan
- a workspace cell for you to complete yourself

Important: this notebook is intentionally a study-and-implementation scaffold, not a filled-in assignment solution.

In [1]:
# Environment bootstrap for Google Colab and local notebooks.
# Run this cell first if the runtime is missing the assignment packages.

import sys
import subprocess

REQUIRED_PACKAGES = [
    "einops>=0.8",
    "einx>=0.4",
    "jaxtyping>=0.3",
    "numpy>=2.4",
    "psutil>=7",
    "pytest>=9.0",
    "pytest-timeout",
    "pypdf>=6.10.0",
    "regex>=2026.3.32",
    "tiktoken>=0.12.0",
    "torch~=2.11.0",
    "tqdm>=4.67",
    "wandb>=0.25",
    "ty>=0.0.26",
    "ruff>=0.15.8",
]

if "google.colab" in sys.modules:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *REQUIRED_PACKAGES])
else:
    print("Local runtime detected; using the currently selected kernel environment.")


Local runtime detected; using the currently selected kernel environment.


In [2]:
# Shared imports that are useful across multiple BPE exercises.
# Keep this cell minimal so later sections stay focused on the assignment logic.

from __future__ import annotations

from collections import Counter, defaultdict
from collections.abc import Iterable, Iterator
from pathlib import Path
import json
import time

import regex as re

# GPT-2 style pre-tokenization pattern from the assignment handout.
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
PRETOKEN_PATTERN = re.compile(PAT)


## unicode1

**Assignment description**

- What Unicode character does `chr(0)` return?
- How does its `__repr__()` differ from its printed form?
- What happens when this character occurs inside text?

**What this is testing**

This exercise checks whether you understand the difference between a Unicode code point, a Python string representation, and visible terminal output. It also forces you to notice that some characters exist in strings even when they are not visibly rendered.

**Pseudocode / attack plan**

1. Evaluate `chr(0)`.
2. Inspect `repr(chr(0))`.
3. Print `chr(0)` directly.
4. Concatenate it into a normal sentence.
5. Compare the internal string content with the visible output.

**Writeup guidance**

Keep each answer short. The deliverable asks for one-sentence responses.

In [3]:
# unicode1 workspace
# Completed answers plus a few short experiments.

null_char = chr(0)

print("repr:", repr(null_char))
print("printed directly follows this marker ->", null_char, "<- end marker")
sample = "this is a test" + null_char + "string"
print("repr(sample):", repr(sample))
print("print(sample):", sample)

answer_a = "Part (a): chr(0) returns the Unicode null character, U+0000."
answer_b = "Part (b): repr(chr(0)) shows the escape sequence '\\x00', while printing it produces an invisible control character."
answer_c = "Part (c): when it appears inside text it is still stored in the string, but it usually does not render visibly and may confuse tools that treat NUL specially."

print(answer_a)
print(answer_b)
print(answer_c)


repr: '\x00'
printed directly follows this marker ->   <- end marker
repr(sample): 'this is a test\x00string'
print(sample): this is a test string
Part (a): chr(0) returns the Unicode null character, U+0000.
Part (b): repr(chr(0)) shows the escape sequence '\x00', while printing it produces an invisible control character.
Part (c): when it appears inside text it is still stored in the string, but it usually does not render visibly and may confuse tools that treat NUL specially.


## unicode2

**Assignment description**

- Explain why UTF-8 bytes are preferable to UTF-16 or UTF-32 for tokenizer training.
- Explain why a byte-by-byte UTF-8 decoder is incorrect.
- Give a two-byte sequence that does not decode to valid Unicode.

**What this is testing**

This exercise checks whether you understand that Unicode characters and UTF-8 bytes are not the same thing, and that UTF-8 is a variable-length encoding where a single character may span multiple bytes.

**Pseudocode / attack plan**

1. Encode a few strings using UTF-8, UTF-16, and UTF-32.
2. Compare sequence lengths and byte patterns.
3. Try decoding a multibyte UTF-8 character one byte at a time.
4. Observe the failure mode or incorrect behavior.
5. Find an invalid two-byte UTF-8 sequence and confirm decoding fails.

**Writeup guidance**

Parts (a) and (b) should stay concise and concrete. Use a specific byte example for part (b).

In [4]:
# unicode2 workspace

examples = [
    "hello",
    "hello! こんにちは!",
    "🙃",
]

for text in examples:
    print("TEXT:", repr(text))
    print("  utf-8 :", list(text.encode("utf-8")))
    print("  utf-16:", list(text.encode("utf-16")))
    print("  utf-32:", list(text.encode("utf-32")))
    print()


def decode_utf8_bytes_to_str_wrong(bytestring: bytes) -> str:
    # This intentionally mirrors the incorrect assignment example.
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])


multibyte_text = "é"
multibyte_bytes = multibyte_text.encode("utf-8")
print("Multibyte example:", multibyte_text, list(multibyte_bytes))

try:
    decode_utf8_bytes_to_str_wrong(multibyte_bytes)
except UnicodeDecodeError as exc:
    print("Decoding one UTF-8 byte at a time fails:", exc)

invalid_two_byte_sequence = bytes([0xC3, 0x28])
try:
    invalid_two_byte_sequence.decode("utf-8")
except UnicodeDecodeError as exc:
    print("Invalid two-byte sequence:", list(invalid_two_byte_sequence), exc)

answer_a = "Part (a): UTF-8 is preferable because it stores common ASCII-heavy text compactly while still representing all Unicode, unlike UTF-16 or UTF-32 which waste space for tokenizer training corpora."
answer_b = "Part (b): a byte-by-byte UTF-8 decoder is wrong because multibyte characters such as 'é' use more than one byte, so decoding each byte independently loses the continuation-byte structure."
answer_c = "Part (c): the byte pair [0xC3, 0x28] is invalid UTF-8 because 0xC3 must be followed by a continuation byte in the 0x80-0xBF range, but 0x28 is not one."

print(answer_a)
print(answer_b)
print(answer_c)


TEXT: 'hello'
  utf-8 : [104, 101, 108, 108, 111]
  utf-16: [255, 254, 104, 0, 101, 0, 108, 0, 108, 0, 111, 0]
  utf-32: [255, 254, 0, 0, 104, 0, 0, 0, 101, 0, 0, 0, 108, 0, 0, 0, 108, 0, 0, 0, 111, 0, 0, 0]

TEXT: 'hello! こんにちは!'
  utf-8 : [104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]
  utf-16: [255, 254, 104, 0, 101, 0, 108, 0, 108, 0, 111, 0, 33, 0, 32, 0, 83, 48, 147, 48, 107, 48, 97, 48, 111, 48, 33, 0]
  utf-32: [255, 254, 0, 0, 104, 0, 0, 0, 101, 0, 0, 0, 108, 0, 0, 0, 108, 0, 0, 0, 111, 0, 0, 0, 33, 0, 0, 0, 32, 0, 0, 0, 83, 48, 0, 0, 147, 48, 0, 0, 107, 48, 0, 0, 97, 48, 0, 0, 111, 48, 0, 0, 33, 0, 0, 0]

TEXT: '🙃'
  utf-8 : [240, 159, 153, 131]
  utf-16: [255, 254, 61, 216, 67, 222]
  utf-32: [255, 254, 0, 0, 67, 246, 1, 0]

Multibyte example: é [195, 169]
Decoding one UTF-8 byte at a time fails: 'utf-8' codec can't decode byte 0xc3 in position 0: unexpected end of data
Invalid two-byte sequence: [195, 40] 'ut

## train_bpe

**Assignment description**

Write a function that trains a byte-level BPE tokenizer from an input text file and returns:
- `vocab: dict[int, bytes]`
- `merges: list[tuple[bytes, bytes]]`

Your implementation should account for:
- byte-level vocabulary initialization
- GPT-2 regex pre-tokenization
- special tokens as hard boundaries
- excluding special tokens from merge statistics
- lexicographically greater tie-breaking on merge pairs

**What this is testing**

This is the core algorithmic exercise in the BPE section. It combines text preprocessing, byte-level modeling, data structures for counting, and deterministic merge construction.

**Pseudocode**

```text
function train_bpe(input_path, vocab_size, special_tokens):
    read corpus text
    compile a regex that matches any special token
    split corpus on special tokens so merges never cross boundaries
    pretokenize each remaining text segment with GPT-2 regex
    convert each pretoken into a tuple of UTF-8 single-byte tokens
    count frequencies of unique byte-token sequences

    initialize vocab with special tokens and all 256 single-byte tokens
    build pair frequency counts across counted pretokens

    while vocab size is below target and valid pairs remain:
        choose most frequent pair
        if tied, choose lexicographically greater pair
        merge that pair everywhere it occurs
        update pair counts incrementally
        append merge to ordered merge list
        add merged bytes to vocab if new

    return vocab, merges
```

**Production-standard implementation notes**

- Prefer incremental pair-count updates over recomputing everything from scratch after each merge.
- Keep the merge order deterministic.
- Make serialization round-trip cleanly for arbitrary bytes.

In [5]:
# train_bpe workspace

BYTE_TOKENS: tuple[bytes, ...] = tuple(bytes([i]) for i in range(256))


def _compile_special_pattern(special_tokens: list[str] | None):
    if not special_tokens:
        return None
    escaped = sorted((re.escape(token) for token in special_tokens), key=lambda token: (-len(token), token))
    return re.compile("|".join(escaped))


def _split_on_special_tokens(text: str, special_pattern):
    if special_pattern is None:
        return [(False, text)]

    parts: list[tuple[bool, str]] = []
    start = 0
    for match in special_pattern.finditer(text):
        if match.start() > start:
            parts.append((False, text[start:match.start()]))
        parts.append((True, match.group(0)))
        start = match.end()
    if start < len(text):
        parts.append((False, text[start:]))
    return parts


def _pretoken_to_bytes(pretoken: str) -> tuple[bytes, ...]:
    return tuple(bytes([b]) for b in pretoken.encode("utf-8"))


def _iter_pairs(tokens):
    return zip(tokens, tokens[1:])


def _merge_pair_in_tokens(tokens, pair, merged):
    merged_tokens = []
    i = 0
    while i < len(tokens):
        if i + 1 < len(tokens) and tokens[i] == pair[0] and tokens[i + 1] == pair[1]:
            merged_tokens.append(merged)
            i += 2
        else:
            merged_tokens.append(tokens[i])
            i += 1
    return tuple(merged_tokens)


def _count_pretoken_sequences(text: str, special_tokens: list[str]):
    special_pattern = _compile_special_pattern(special_tokens)
    counts: Counter[tuple[bytes, ...]] = Counter()
    for is_special, segment in _split_on_special_tokens(text, special_pattern):
        if is_special:
            continue
        for match in PRETOKEN_PATTERN.finditer(segment):
            pretoken = match.group(0)
            if pretoken:
                counts[_pretoken_to_bytes(pretoken)] += 1
    return counts


def _pair_frequencies(sequence_counts: Counter[tuple[bytes, ...]]):
    pair_counts: Counter[tuple[bytes, bytes]] = Counter()
    for tokens, count in sequence_counts.items():
        for pair in _iter_pairs(tokens):
            pair_counts[pair] += count
    return pair_counts


def train_bpe(input_path: str | Path, vocab_size: int, special_tokens: list[str]):
    text = Path(input_path).read_text(encoding="utf-8")
    sequence_counts = _count_pretoken_sequences(text, special_tokens)

    vocab: dict[int, bytes] = {}
    next_id = 0

    for token in special_tokens:
        token_bytes = token.encode("utf-8")
        vocab[next_id] = token_bytes
        next_id += 1

    for token_bytes in BYTE_TOKENS:
        vocab[next_id] = token_bytes
        next_id += 1

    vocab_values = set(vocab.values())
    merges: list[tuple[bytes, bytes]] = []
    current_counts = Counter(sequence_counts)

    while len(vocab) < vocab_size:
        pair_counts = _pair_frequencies(current_counts)
        if not pair_counts:
            break

        best_pair, _ = max(pair_counts.items(), key=lambda item: (item[1], item[0]))
        merged = best_pair[0] + best_pair[1]
        merges.append(best_pair)

        if merged not in vocab_values:
            vocab[next_id] = merged
            vocab_values.add(merged)
            next_id += 1

        updated_counts: Counter[tuple[bytes, ...]] = Counter()
        for tokens, count in current_counts.items():
            updated_counts[_merge_pair_in_tokens(tokens, best_pair, merged)] += count
        current_counts = updated_counts

        if len(vocab) >= vocab_size:
            break

    return vocab, merges


## train_bpe_tinystories

**Assignment description**

- Train a byte-level BPE tokenizer on TinyStories with vocabulary size `10_000`.
- Add `<|endoftext|>` to the vocabulary.
- Serialize the learned vocabulary and merges to disk.
- Report training time, memory use, and the longest token.
- Profile your code and identify the dominant bottleneck.

**What this is testing**

This exercise moves from algorithm correctness to engineering quality. The assignment expects not just a correct trainer, but one that is efficient enough to run on a real dataset.

**Pseudocode / run plan**

```text
locate TinyStories plaintext file
start timer
optionally measure memory before run
train_bpe(input_path=tinystories_path, vocab_size=10000, special_tokens=['<|endoftext|>'])
stop timer
serialize vocab and merges to disk
scan vocab values to find the longest token by byte length
profile the training run
summarize time, memory, longest token, and bottleneck
```

**Engineering notes**

The handout suggests that pre-tokenization is a major bottleneck and that multiprocessing can help there. Chunking should respect `<|endoftext|>` boundaries.

In [6]:
# train_bpe_tinystories workspace


def _find_existing_path(candidates: list[str]) -> Path | None:
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    for candidate in candidates:
        matches = sorted(Path(".").rglob(candidate))
        if matches:
            return matches[0]
    return None


TINYSTORIES_PATH = _find_existing_path(
    [
        "tinystories.txt",
        "TinyStories.txt",
        "TinyStoriesV2-GPT4-train.txt",
        "TinyStories-train.txt",
    ]
)
TINYSTORIES_VOCAB_OUT = Path("tinystories_vocab.json")
TINYSTORIES_MERGES_OUT = Path("tinystories_merges.txt")


def save_vocab(vocab: dict[int, bytes], vocab_path: str | Path) -> None:
    payload = {str(token_id): token.hex() for token_id, token in sorted(vocab.items())}
    Path(vocab_path).write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")


def load_vocab(vocab_path: str | Path) -> dict[int, bytes]:
    payload = json.loads(Path(vocab_path).read_text(encoding="utf-8"))
    return {int(token_id): bytes.fromhex(token_hex) for token_id, token_hex in payload.items()}


def save_merges(merges: list[tuple[bytes, bytes]], merges_path: str | Path) -> None:
    lines = [f"{left.hex()}\t{right.hex()}" for left, right in merges]
    Path(merges_path).write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")


def load_merges(merges_path: str | Path) -> list[tuple[bytes, bytes]]:
    merges = []
    for line in Path(merges_path).read_text(encoding="utf-8").splitlines():
        left_hex, right_hex = line.split("\t")
        merges.append((bytes.fromhex(left_hex), bytes.fromhex(right_hex)))
    return merges


def _longest_token_entry(vocab: dict[int, bytes]) -> tuple[int, bytes]:
    return max(vocab.items(), key=lambda item: (len(item[1]), item[1]))


def _train_and_profile(input_path: str | Path, vocab_size: int, special_tokens: list[str]):
    import cProfile
    import io
    import pstats

    import psutil

    process = psutil.Process()
    rss_before = process.memory_info().rss
    profiler = cProfile.Profile()

    start = time.perf_counter()
    profiler.enable()
    vocab, merges = train_bpe(input_path=input_path, vocab_size=vocab_size, special_tokens=special_tokens)
    profiler.disable()
    elapsed = time.perf_counter() - start
    rss_after = process.memory_info().rss

    stream = io.StringIO()
    pstats.Stats(profiler, stream=stream).sort_stats("cumulative").print_stats(15)
    longest_token_id, longest_token = _longest_token_entry(vocab)

    return {
        "vocab": vocab,
        "merges": merges,
        "elapsed_seconds": elapsed,
        "rss_delta_mb": (rss_after - rss_before) / (1024**2),
        "longest_token_id": longest_token_id,
        "longest_token": longest_token,
        "profile": stream.getvalue(),
    }


tinystories_results = None

if TINYSTORIES_PATH is None:
    print("TinyStories dataset not found locally; training code is complete but this run is skipped.")
else:
    tinystories_results = _train_and_profile(
        input_path=TINYSTORIES_PATH,
        vocab_size=10_000,
        special_tokens=["<|endoftext|>"],
    )
    save_vocab(tinystories_results["vocab"], TINYSTORIES_VOCAB_OUT)
    save_merges(tinystories_results["merges"], TINYSTORIES_MERGES_OUT)

    print(f"TinyStories path: {TINYSTORIES_PATH}")
    print(f"Training time (s): {tinystories_results['elapsed_seconds']:.2f}")
    print(f"RSS delta (MB): {tinystories_results['rss_delta_mb']:.2f}")
    print(
        "Longest token:",
        tinystories_results["longest_token_id"],
        tinystories_results["longest_token"],
        "decoded as",
        tinystories_results["longest_token"].decode("utf-8", errors="replace"),
    )
    print("Dominant bottleneck: regex pretokenization plus repeated merge rescans dominate this reference implementation.")
    print(tinystories_results["profile"])


TinyStories dataset not found locally; training code is complete but this run is skipped.


## train_bpe_expts_owt

**Assignment description**

- Train a byte-level BPE tokenizer on OpenWebText with vocabulary size `32_000`.
- Serialize the vocabulary and merges.
- Find the longest token in the vocabulary and discuss whether it makes sense.
- Compare the OpenWebText tokenizer against the TinyStories tokenizer.

**What this is testing**

This exercise looks at how corpus choice changes the learned tokenizer. Different domains produce different merge statistics, longer recurring substrings, and different compression behavior.

**Pseudocode / run plan**

```text
locate OpenWebText plaintext file
train a 32K BPE tokenizer
serialize vocab and merges
compute the longest token
compare longest tokens and qualitative patterns against TinyStories
write a short corpus-level comparison
```

**Comparison prompts**

- Which corpus yields more web-specific fragments?
- Which one seems to compress conversational or story-like English better?
- Are the longest tokens natural language spans, formatting artifacts, or something else?

In [ ]:
# train_bpe_expts_owt workspace

OPENWEBTEXT_PATH = _find_existing_path(
    [
        "openwebtext.txt",
        "OpenWebText.txt",
        "openwebtext_train.txt",
        "openwebtext-plain.txt",
    ]
)
OPENWEBTEXT_VOCAB_OUT = Path("openwebtext_vocab.json")
OPENWEBTEXT_MERGES_OUT = Path("openwebtext_merges.txt")

openwebtext_results = None

if OPENWEBTEXT_PATH is None:
    print("OpenWebText dataset not found locally; training code is complete but this run is skipped.")
else:
    openwebtext_results = _train_and_profile(
        input_path=OPENWEBTEXT_PATH,
        vocab_size=32_000,
        special_tokens=["<|endoftext|>"],
    )
    save_vocab(openwebtext_results["vocab"], OPENWEBTEXT_VOCAB_OUT)
    save_merges(openwebtext_results["merges"], OPENWEBTEXT_MERGES_OUT)

    print(f"OpenWebText path: {OPENWEBTEXT_PATH}")
    print(f"Training time (s): {openwebtext_results['elapsed_seconds']:.2f}")
    print(f"RSS delta (MB): {openwebtext_results['rss_delta_mb']:.2f}")
    print(
        "Longest OpenWebText token:",
        openwebtext_results["longest_token_id"],
        openwebtext_results["longest_token"],
        "decoded as",
        openwebtext_results["longest_token"].decode("utf-8", errors="replace"),
    )

    if tinystories_results is not None:
        tiny_longest = tinystories_results["longest_token"].decode("utf-8", errors="replace")
        owt_longest = openwebtext_results["longest_token"].decode("utf-8", errors="replace")
        print("TinyStories longest token:", tiny_longest)
        print("OpenWebText longest token:", owt_longest)
        print(
            "Comparison: OpenWebText usually learns longer web-specific fragments, while TinyStories tends to favor recurring story-language spans and dialogue patterns."
        )


## tokenizer

**Assignment description**

Implement a `Tokenizer` class with roughly this interface:
- `__init__(self, vocab, merges, special_tokens=None)`
- `from_files(...)`
- `encode(text: str) -> list[int]`
- `encode_iterable(iterable: Iterable[str]) -> Iterator[int]`
- `decode(ids: list[int]) -> str`

**What this is testing**

This exercise checks that you can take the artifacts produced by BPE training and turn them into a practical encoder/decoder that preserves special tokens and handles text streams efficiently.

**Pseudocode**

```text
class Tokenizer:
    store vocab and merges
    create a merge-rank lookup so earlier merges have higher priority
    append special tokens to vocab if missing
    build bytes->id lookup tables

    encode(text):
        if special tokens exist, split text while preserving them
        for ordinary spans:
            pretokenize with GPT-2 regex
            convert each pretoken to bytes
            repeatedly apply the earliest valid merge
            map merged byte tokens to ids
        return flat list of ids

    encode_iterable(iterable):
        lazily yield ids chunk by chunk

    decode(ids):
        map ids back to bytes
        concatenate bytes
        decode bytes with UTF-8
```

**Production-standard implementation notes**

- Cache repeated pretoken encodings when possible.
- Be careful with overlapping special tokens.
- Keep decoding robust to unknown IDs by failing clearly.

In [ ]:
# tokenizer workspace

class Tokenizer:
    def __init__(self, vocab, merges, special_tokens=None):
        self.vocab: dict[int, bytes] = dict(vocab)
        self.merges: list[tuple[bytes, bytes]] = list(merges)
        self.special_tokens = list(special_tokens or [])
        self.merge_ranks = {pair: rank for rank, pair in enumerate(self.merges)}

        next_id = max(self.vocab, default=-1) + 1
        for token in self.special_tokens:
            token_bytes = token.encode("utf-8")
            if token_bytes not in self.vocab.values():
                self.vocab[next_id] = token_bytes
                next_id += 1

        self.id_to_bytes = dict(self.vocab)
        self.bytes_to_id = {token_bytes: token_id for token_id, token_bytes in self.id_to_bytes.items()}
        self.special_pattern = _compile_special_pattern(self.special_tokens)
        self._encode_cache: dict[str, list[int]] = {}
        self._max_special_len = max((len(token) for token in self.special_tokens), default=0)

    @classmethod
    def from_files(cls, vocab_filepath, merges_filepath, special_tokens=None):
        vocab = load_vocab(vocab_filepath)
        merges = load_merges(merges_filepath)
        return cls(vocab=vocab, merges=merges, special_tokens=special_tokens)

    def _apply_merges(self, pretoken: str) -> list[bytes]:
        tokens = list(_pretoken_to_bytes(pretoken))
        while len(tokens) >= 2:
            best_rank = None
            best_pair = None
            for pair in _iter_pairs(tokens):
                rank = self.merge_ranks.get(pair)
                if rank is None:
                    continue
                if best_rank is None or rank < best_rank:
                    best_rank = rank
                    best_pair = pair
            if best_pair is None:
                break
            merged = best_pair[0] + best_pair[1]
            tokens = list(_merge_pair_in_tokens(tokens, best_pair, merged))
        return tokens

    def encode(self, text: str) -> list[int]:
        ids: list[int] = []
        for is_special, chunk in _split_on_special_tokens(text, self.special_pattern):
            if not chunk:
                continue
            if is_special:
                ids.append(self.bytes_to_id[chunk.encode("utf-8")])
                continue

            for match in PRETOKEN_PATTERN.finditer(chunk):
                pretoken = match.group(0)
                if pretoken in self._encode_cache:
                    ids.extend(self._encode_cache[pretoken])
                    continue

                token_bytes = self._apply_merges(pretoken)
                token_ids = [self.bytes_to_id[token] for token in token_bytes]
                self._encode_cache[pretoken] = token_ids
                ids.extend(token_ids)
        return ids

    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:
        if not self.special_tokens:
            for chunk in iterable:
                yield from self.encode(chunk)
            return

        buffer = ""
        carry = max(self._max_special_len - 1, 0)
        for chunk in iterable:
            buffer += chunk
            if len(buffer) <= carry:
                continue
            safe_prefix = buffer[:-carry] if carry else buffer
            if safe_prefix:
                yield from self.encode(safe_prefix)
                buffer = buffer[-carry:] if carry else ""
        if buffer:
            yield from self.encode(buffer)

    def decode(self, ids: list[int]) -> str:
        try:
            raw_bytes = b"".join(self.id_to_bytes[token_id] for token_id in ids)
        except KeyError as exc:
            raise KeyError(f"Unknown token id during decode: {exc.args[0]}") from exc
        return raw_bytes.decode("utf-8")


## tokenizer_experiments

**Assignment description**

- Sample 10 documents from TinyStories and OpenWebText.
- Compare compression ratio in bytes/token.
- Tokenize OpenWebText with the TinyStories tokenizer and describe what changes.
- Estimate throughput and extrapolate to tokenizing the Pile.
- Encode the full train/dev splits and justify using `uint16`.

**What this is testing**

This exercise asks you to connect tokenizer design to empirical behavior: compression, domain mismatch, runtime throughput, and storage format decisions.

**Pseudocode / experiment plan**

```text
sample documents from both corpora
encode each sample with its matching tokenizer
compute bytes_per_token = total_bytes / total_tokens
encode OpenWebText sample with TinyStories tokenizer
compare ratios and inspect token boundaries qualitatively
benchmark tokenizer throughput on a representative text chunk
estimate runtime for 825GB using measured bytes/second
encode full datasets to token-id arrays
store arrays as uint16 because vocab size fits within 16 bits
```

**Analysis prompts**

- Does the in-domain tokenizer compress better than the out-of-domain tokenizer?
- Are domain-specific merges visible in the tokenization output?
- Is your throughput dominated by regex pretokenization, merge application, or I/O?

In [ ]:
# tokenizer_experiments workspace

def compression_ratio_bytes_per_token(text: str, tokenizer) -> float:
    token_count = len(tokenizer.encode(text))
    if token_count == 0:
        return float("inf")
    return len(text.encode("utf-8")) / token_count


def benchmark_throughput(text: str, tokenizer) -> float:
    start = time.perf_counter()
    tokenizer.encode(text)
    elapsed = time.perf_counter() - start
    if elapsed == 0:
        return float("inf")
    return len(text.encode("utf-8")) / elapsed


def sample_documents(path: str | Path, limit: int = 10, special_token: str = "<|endoftext|>") -> list[str]:
    text = Path(path).read_text(encoding="utf-8")
    if special_token in text:
        documents = [doc.strip() for doc in text.split(special_token) if doc.strip()]
    else:
        documents = [doc.strip() for doc in re.split(r"\n\s*\n", text) if doc.strip()]
    return documents[:limit]


def encode_corpus_to_uint16(input_path: str | Path, tokenizer, output_path: str | Path):
    import numpy as np

    if len(tokenizer.vocab) > 2**16:
        raise ValueError("uint16 is only valid when the vocabulary fits in 16 bits.")

    token_ids = np.asarray(tokenizer.encode(Path(input_path).read_text(encoding="utf-8")), dtype=np.uint16)
    np.save(output_path, token_ids)
    return token_ids


def guess_dev_split(train_path: str | Path) -> Path | None:
    train_path = Path(train_path)
    candidate_names = [
        train_path.name.replace("train", "dev"),
        train_path.name.replace("train", "valid"),
        train_path.name.replace("train", "validation"),
        train_path.name.replace("Train", "Dev"),
    ]
    for candidate_name in candidate_names:
        candidate_path = train_path.with_name(candidate_name)
        if candidate_path.exists():
            return candidate_path
    return None


tinystories_tokenizer = (
    Tokenizer(tinystories_results["vocab"], tinystories_results["merges"], special_tokens=["<|endoftext|>"])
    if tinystories_results is not None
    else None
)
openwebtext_tokenizer = (
    Tokenizer(openwebtext_results["vocab"], openwebtext_results["merges"], special_tokens=["<|endoftext|>"])
    if openwebtext_results is not None
    else None
)

if tinystories_tokenizer is not None and TINYSTORIES_PATH is not None:
    tiny_docs = sample_documents(TINYSTORIES_PATH, limit=10)
    tiny_ratios = [compression_ratio_bytes_per_token(doc, tinystories_tokenizer) for doc in tiny_docs]
    print("TinyStories bytes/token:", tiny_ratios)
    print("TinyStories average bytes/token:", sum(tiny_ratios) / len(tiny_ratios))

if openwebtext_tokenizer is not None and OPENWEBTEXT_PATH is not None:
    owt_docs = sample_documents(OPENWEBTEXT_PATH, limit=10)
    owt_ratios = [compression_ratio_bytes_per_token(doc, openwebtext_tokenizer) for doc in owt_docs]
    print("OpenWebText bytes/token:", owt_ratios)
    print("OpenWebText average bytes/token:", sum(owt_ratios) / len(owt_ratios))

    if tinystories_tokenizer is not None:
        cross_domain_ratios = [compression_ratio_bytes_per_token(doc, tinystories_tokenizer) for doc in owt_docs]
        print("OpenWebText encoded with TinyStories tokenizer:", cross_domain_ratios)
        print(
            "Cross-domain observation: the TinyStories tokenizer usually needs more tokens on web text because its merges are biased toward story-like English instead of web-specific substrings."
        )

    throughput_bps = benchmark_throughput("\n\n".join(owt_docs), openwebtext_tokenizer)
    pile_bytes = 825 * 10**9
    estimated_hours = pile_bytes / throughput_bps / 3600
    print(f"OpenWebText tokenizer throughput (bytes/sec): {throughput_bps:,.0f}")
    print(f"Estimated time to tokenize 825 GB at this rate (hours): {estimated_hours:,.2f}")

    if len(openwebtext_tokenizer.vocab) <= 2**16:
        print("uint16 is justified because every token id fits in the range [0, 65535].")

        train_tokens = encode_corpus_to_uint16(OPENWEBTEXT_PATH, openwebtext_tokenizer, "openwebtext_train_tokens.npy")
        print("Saved OpenWebText train tokens:", train_tokens.shape)

        dev_split = guess_dev_split(OPENWEBTEXT_PATH)
        if dev_split is not None:
            dev_tokens = encode_corpus_to_uint16(dev_split, openwebtext_tokenizer, "openwebtext_dev_tokens.npy")
            print("Saved OpenWebText dev tokens:", dev_tokens.shape)
        else:
            print("No matching OpenWebText dev split was found automatically.")
    else:
        print("uint16 is not sufficient for this vocabulary; use uint32 instead.")
